# Phase 3.1 — Recommendation Baselines

Build simple recommendation baselines that provide a reference point for later content-based, collaborative, and hybrid models.

Baselines:
1. Popularity-based recommendation.
2. Simple item-to-item co-occurrence recommendation.
3. Top-K offline evaluation.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation_interactions.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

K = 10

print("Processed directory:", PROCESSED_DIR)


Processed directory: f:\annuspeaks.com\recommendation-system\data\processed


## 3.1.1 Load Evaluation Data

The training split is used to build the baselines. Validation/test interactions are kept unseen during recommendation generation.


In [2]:
columns = [
    "user_id",
    "item_id",
    "interaction_type",
    "weight",
    "timestamp",
]

train = pd.read_csv(TRAIN_PATH, usecols=columns)
validation = pd.read_csv(VALIDATION_PATH, usecols=columns)
test = pd.read_csv(TEST_PATH, usecols=columns)

print("Train:", f"{len(train):,}")
print("Validation:", f"{len(validation):,}")
print("Test:", f"{len(test):,}")


Train: 2,356,045
Validation: 200,028
Test: 200,028


## 3.1.2 Popularity Baseline

Rank products by their total behavioral preference weight in the training data.

This is the simplest production-style fallback and does not require user-specific modeling.


In [3]:
# Build the popularity ranking from training data only.

popularity = (
    train.groupby("item_id")
    .agg(
        total_weight=("weight", "sum"),
        interaction_count=("item_id", "size"),
    )
    .sort_values(
        ["total_weight", "interaction_count"],
        ascending=False,
    )
)

popular_items = popularity.index.tolist()

print("Ranked products:", f"{len(popular_items):,}")
display(popularity.head(10))


Ranked products: 228,392


,total_weight,interaction_count
item_id,,
461686,3287,2483
187946,3221,3217
5411,2192,2178
370653,1724,1724
219512,1529,1437
257040,1527,1331
320130,1507,1207
7943,1492,1278
96924,1472,1472


In [4]:
def popularity_recommend(user_id, k=10):
    return popular_items[:k]

print("Popularity recommendation example:")
print(popularity_recommend(1, K))


Popularity recommendation example:
[461686, 187946, 5411, 370653, 219512, 257040, 320130, 7943, 96924, 298009]


## 3.1.3 Simple Similar-Item Baseline

Build a lightweight item-to-item co-occurrence model from training histories.

For computational safety, each user's history is limited to their most recent 20 unique products when constructing co-occurrence pairs. This is a baseline, not the final similarity model.


In [5]:
# Prepare bounded user histories for co-occurrence modeling.

MAX_HISTORY_ITEMS = 20
user_histories = {}

train_sorted = train.sort_values(
    ["user_id", "timestamp"],
    kind="mergesort",
)

for user_id, group in train_sorted.groupby("user_id", sort=False):
    items = group["item_id"].drop_duplicates().tolist()
    user_histories[user_id] = items[-MAX_HISTORY_ITEMS:]

print("User histories:", f"{len(user_histories):,}")


User histories: 1,407,580


In [6]:
# Build bounded item-item co-occurrence counts.

cooccurrence = defaultdict(Counter)

for items in user_histories.values():
    if len(items) < 2:
        continue

    for i, item_a in enumerate(items):
        for item_b in items[i + 1:]:
            cooccurrence[item_a][item_b] += 1
            cooccurrence[item_b][item_a] += 1

print("Items with similarity relationships:", f"{len(cooccurrence):,}")


Items with similarity relationships: 127,400


In [7]:
def similar_item_recommend(user_id, k=10):
    history = user_histories.get(user_id, [])

    if not history:
        return popular_items[:k]

    scores = Counter()

    # Use the user's most recent interacted items as recommendation seeds.
    for item_id in reversed(history[-5:]):
        for candidate, count in cooccurrence.get(item_id, {}).items():
            if candidate not in history:
                scores[candidate] += count

    if not scores:
        return popular_items[:k]

    return [
        item_id
        for item_id, _ in scores.most_common(k)
    ]

example_user = next(iter(user_histories))

print("User:", example_user)
print("History:", user_histories[example_user][-10:])
print("Recommendations:", similar_item_recommend(example_user, K))


User: 0
History: [285930]
Recommendations: [124804, 8692, 314688, 453814, 203022, 365536, 204776, 365673, 399674, 159038]


## 3.1.4 Top-K Evaluation

Evaluate whether the held-out test item appears in the Top-K recommendations.

Because the temporal split assigns one final interaction to test for users with sufficient history, each eligible user has a held-out target.


In [8]:
def build_test_targets(test_data):
    return (
        test_data
        .groupby("user_id")["item_id"]
        .last()
        .to_dict()
    )

test_targets = build_test_targets(test)

print("Test users:", f"{len(test_targets):,}")


Test users: 200,028


In [9]:
def evaluate_recommender(recommender, targets, k=10):
    hits = 0
    evaluated = 0

    for user_id, target_item in targets.items():
        recommendations = recommender(user_id, k)

        if target_item in recommendations:
            hits += 1

        evaluated += 1

    hit_rate = hits / evaluated if evaluated else 0.0

    return {
        "K": k,
        "evaluated_users": evaluated,
        "hits": hits,
        "HitRate@K": hit_rate,
    }

popularity_result = evaluate_recommender(
    popularity_recommend,
    test_targets,
    K,
)

similar_item_result = evaluate_recommender(
    similar_item_recommend,
    test_targets,
    K,
)

results = pd.DataFrame([
    {"baseline": "Popularity", **popularity_result},
    {"baseline": "Similar-Item", **similar_item_result},
])

display(results)


,baseline,K,evaluated_users,hits,HitRate@K
0,Popularity,10,200028,1281,0.006404
1,Similar-Item,10,200028,20634,0.103156


## 3.1.5 Baseline Comparison

The popularity baseline establishes the minimum reference performance.

The similar-item baseline provides a simple personalized reference based on item co-occurrence.

These results will be used as the benchmark for later recommendation models.


In [10]:
# Final Phase 3.1 validation

assert len(popular_items) > 0
assert len(test_targets) > 0
assert set(results["baseline"]) == {"Popularity", "Similar-Item"}
assert results["HitRate@K"].between(0, 1).all()

print("Phase 3.1 validation: PASS")
display(results)


Phase 3.1 validation: PASS


,baseline,K,evaluated_users,hits,HitRate@K
0,Popularity,10,200028,1281,0.006404
1,Similar-Item,10,200028,20634,0.103156


## Phase 3.1 Completion

- Popularity baseline built from training data.
- Simple similar-item co-occurrence baseline built from training histories.
- Both baselines evaluated with HitRate@10 on the held-out test set.
